In [ ]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.final_aligned_observation_writer import build_final_aligned_observations_stage


In [ ]:
engine = get_engine_from_env()

In [ ]:
result = build_final_aligned_observations_stage(
    engine=engine,
    schema="capstone",
    premelt_table="synthetic_observations_premelt_stage",
    rebuilt_table="synthetic_sensor_observations_rebuilt_stage",
    target_table="synthetic_sensor_observations_final_aligned_stage",
    dataset_id="pump_synthetic_v1",
    run_id="premelt_run_001",
    n_sensors=52,
    complete_only=True,
    prefer_rebuilt_sensor_values=True,
    if_exists="replace",
)


In [ ]:

result

----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS row_count,
        MIN(batch_id) AS min_batch_id,
        MAX(batch_id) AS max_batch_id,
        MIN(row_in_batch) AS min_row_in_batch,
        MAX(observation_index) AS max_observation_index,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_rows,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp
    FROM capstone.synthetic_sensor_observations_final_aligned_stage
    """
)


In [ ]:

display(validation_dataframe)

----

In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        batch_id,
        row_in_batch,
        global_cycle_id,
        stream_state,
        phase,
        created_at,
        meta_episode_id,
        meta_primary_fault_type,
        meta_magnitude,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        dataset_id,
        run_id,
        asset_id,
        generated_row_id,
        observation_index,
        observation_timestamp,
        rebuild_is_complete
    FROM capstone.synthetic_sensor_observations_final_aligned_stage
    ORDER BY batch_id, row_in_batch
    LIMIT 10
    """
)


In [ ]:

display(sample_dataframe)